In [1]:
import pandas as pd
import pickle
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn_quantile import RandomForestQuantileRegressor
from tqdm.auto import tqdm
import CRPS.CRPS as pscore


import multiprocessing as mp
mp.set_start_method('spawn')


import sys
sys.path.append('../../../TaskExecutionTimeMining/')
from quantile_regression import QuantileRegression


sys.path.append('../../../Evaluation/')
import conduct_evaluation
from normal_evaluation.quantile_regression_evaluation import *
from normal_evaluation.normal_evaluation import SampleOutcomes_Normal

get_pscores = lambda likelihoods : [pscore(likelihoods[1][i], likelihoods[2][k][3]).compute()[0] for i, k in enumerate(list(likelihoods[0].keys()))]


In [2]:
with open('./quantile_regression_models.pkl', 'rb') as f:
    quantile_regression_models = pickle.load(f)

In [3]:
with open('../../transformed_event_logs/BPIC_2017_all_test.pickle', 'rb') as f:
    test_data = pickle.load(f)

activity_count = [
 'W_Assess potential fraud__ate_abort',
 'W_Assess potential fraud__complete',
 'W_Assess potential fraud__resume',
 'W_Assess potential fraud__schedule',
 'W_Assess potential fraud__start',
 'W_Assess potential fraud__suspend',
 'W_Assess potential fraud__withdraw',
 'W_Call after offers__ate_abort',
 'W_Call after offers__complete',
 'W_Call after offers__resume',
 'W_Call after offers__schedule',
 'W_Call after offers__start',
 'W_Call after offers__suspend',
 'W_Call after offers__withdraw',
 'W_Call incomplete files__ate_abort',
 'W_Call incomplete files__complete',
 'W_Call incomplete files__resume',
 'W_Call incomplete files__schedule',
 'W_Call incomplete files__start',
 'W_Call incomplete files__suspend',
 'W_Complete application__ate_abort',
 'W_Complete application__complete',
 'W_Complete application__resume',
 'W_Complete application__schedule',
 'W_Complete application__start',
 'W_Complete application__suspend',
 'W_Handle leads__complete',
 'W_Handle leads__resume',
 'W_Handle leads__schedule',
 'W_Handle leads__start',
 'W_Handle leads__suspend',
 'W_Handle leads__withdraw',
 'W_Shortened completion __resume',
 'W_Shortened completion __schedule',
 'W_Shortened completion __start',
 'W_Shortened completion __suspend',
 'W_Validate application__ate_abort',
 'W_Validate application__complete',
 'W_Validate application__resume',
 'W_Validate application__schedule',
 'W_Validate application__start',
 'W_Validate application__suspend',
 ]

resource_count = [
'User_1',
 'User_10',
 'User_100',
 'User_101',
 'User_102',
 'User_103',
 'User_104',
 'User_105',
 'User_106',
 'User_107',
 'User_108',
 'User_109',
 'User_11',
 'User_110',
 'User_111',
 'User_112',
 'User_113',
 'User_114',
 'User_115',
 'User_116',
 'User_117',
 'User_118',
 'User_119',
 'User_12',
 'User_120',
 'User_121',
 'User_122',
 'User_123',
 'User_124',
 'User_125',
 'User_126',
 'User_127',
 'User_128',
 'User_129',
 'User_13',
 'User_130',
 'User_131',
 'User_132',
 'User_133',
 'User_134',
 'User_135',
 'User_136',
 'User_137',
 'User_138',
 'User_139',
 'User_14',
 'User_140',
 'User_141',
 'User_142',
 'User_143',
 'User_144',
 'User_145',
 'User_146',
 'User_147',
 'User_148',
 'User_149',
 'User_15',
 'User_16',
 'User_17',
 'User_18',
 'User_19',
 'User_2',
 'User_20',
 'User_21',
 'User_22',
 'User_23',
 'User_24',
 'User_25',
 'User_26',
 'User_27',
 'User_28',
 'User_29',
 'User_3',
 'User_30',
 'User_31',
 'User_32',
 'User_33',
 'User_34',
 'User_35',
 'User_36',
 'User_37',
 'User_38',
 'User_39',
 'User_4',
 'User_40',
 'User_41',
 'User_42',
 'User_43',
 'User_44',
 'User_45',
 'User_46',
 'User_47',
 'User_48',
 'User_49',
 'User_5',
 'User_50',
 'User_51',
 'User_52',
 'User_53',
 'User_54',
 'User_55',
 'User_56',
 'User_57',
 'User_58',
 'User_59',
 'User_6',
 'User_60',
 'User_61',
 'User_62',
 'User_63',
 'User_64',
 'User_65',
 'User_66',
 'User_67',
 'User_68',
 'User_69',
 'User_7',
 'User_70',
 'User_71',
 'User_72',
 'User_73',
 'User_74',
 'User_75',
 'User_76',
 'User_77',
 'User_78',
 'User_79',
 'User_8',
 'User_80',
 'User_81',
 'User_82',
 'User_83',
 'User_84',
 'User_85',
 'User_86',
 'User_87',
 'User_88',
 'User_89',
 'User_9',
 'User_90',
 'User_91',
 'User_92',
 'User_93',
 'User_94',
 'User_95',
 'User_96',
 'User_97',
 'User_98',
 'User_99',
]

ii1 = [
    'intercase_n_1__W_Assess potential fraud__ate_abort',
 'intercase_n_1__W_Assess potential fraud__complete',
 'intercase_n_1__W_Assess potential fraud__resume',
 'intercase_n_1__W_Assess potential fraud__schedule',
 'intercase_n_1__W_Assess potential fraud__start',
 'intercase_n_1__W_Assess potential fraud__suspend',
 'intercase_n_1__W_Assess potential fraud__withdraw',
 'intercase_n_1__W_Call after offers__ate_abort',
 'intercase_n_1__W_Call after offers__complete',
 'intercase_n_1__W_Call after offers__resume',
 'intercase_n_1__W_Call after offers__schedule',
 'intercase_n_1__W_Call after offers__start',
 'intercase_n_1__W_Call after offers__suspend',
 'intercase_n_1__W_Call after offers__withdraw',
 'intercase_n_1__W_Call incomplete files__ate_abort',
 'intercase_n_1__W_Call incomplete files__complete',
 'intercase_n_1__W_Call incomplete files__resume',
 'intercase_n_1__W_Call incomplete files__schedule',
 'intercase_n_1__W_Call incomplete files__start',
 'intercase_n_1__W_Call incomplete files__suspend',
 'intercase_n_1__W_Complete application__ate_abort',
 'intercase_n_1__W_Complete application__complete',
 'intercase_n_1__W_Complete application__resume',
 'intercase_n_1__W_Complete application__schedule',
 'intercase_n_1__W_Complete application__start',
 'intercase_n_1__W_Complete application__suspend',
 'intercase_n_1__W_Handle leads__complete',
 'intercase_n_1__W_Handle leads__resume',
 'intercase_n_1__W_Handle leads__schedule',
 'intercase_n_1__W_Handle leads__start',
 'intercase_n_1__W_Handle leads__suspend',
 'intercase_n_1__W_Handle leads__withdraw',
 'intercase_n_1__W_Shortened completion __resume',
 'intercase_n_1__W_Shortened completion __schedule',
 'intercase_n_1__W_Shortened completion __start',
 'intercase_n_1__W_Shortened completion __suspend',
 'intercase_n_1__W_Validate application__ate_abort',
 'intercase_n_1__W_Validate application__complete',
 'intercase_n_1__W_Validate application__resume',
 'intercase_n_1__W_Validate application__schedule',
 'intercase_n_1__W_Validate application__start',
 'intercase_n_1__W_Validate application__suspend',
]

ii3 = [
    'intercase_n_3__W_Assess potential fraud__ate_abort_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Assess potential fraud__ate_abort_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Assess potential fraud__complete_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Assess potential fraud__complete_W_Complete application__schedule_W_Complete application__start',
 'intercase_n_3__W_Assess potential fraud__complete_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Assess potential fraud__resume_W_Assess potential fraud__complete_W_Validate application__schedule',
 'intercase_n_3__W_Assess potential fraud__resume_W_Assess potential fraud__start_W_Assess potential fraud__suspend',
 'intercase_n_3__W_Assess potential fraud__resume_W_Assess potential fraud__start_W_Call after offers__schedule',
 'intercase_n_3__W_Assess potential fraud__resume_W_Assess potential fraud__start_W_Handle leads__schedule',
 'intercase_n_3__W_Assess potential fraud__resume_W_Assess potential fraud__start_W_Validate application__schedule',
 'intercase_n_3__W_Assess potential fraud__resume_W_Assess potential fraud__suspend_W_Assess potential fraud__resume',
 'intercase_n_3__W_Assess potential fraud__resume_W_Assess potential fraud__suspend_W_Complete application__schedule',
 'intercase_n_3__W_Assess potential fraud__resume_W_Assess potential fraud__suspend_W_Validate application__schedule',
 'intercase_n_3__W_Assess potential fraud__resume_W_Call after offers__schedule_W_Call after offers__start',
 'intercase_n_3__W_Assess potential fraud__resume_W_Complete application__schedule_W_Complete application__start',
 'intercase_n_3__W_Assess potential fraud__resume_W_Handle leads__schedule_W_Handle leads__start',
 'intercase_n_3__W_Assess potential fraud__resume_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Assess potential fraud__schedule_W_Assess potential fraud__start_W_Assess potential fraud__complete',
 'intercase_n_3__W_Assess potential fraud__schedule_W_Assess potential fraud__start_W_Assess potential fraud__suspend',
 'intercase_n_3__W_Assess potential fraud__schedule_W_Assess potential fraud__start_W_Call after offers__schedule',
 'intercase_n_3__W_Assess potential fraud__schedule_W_Assess potential fraud__start_W_Handle leads__schedule',
 'intercase_n_3__W_Assess potential fraud__schedule_W_Assess potential fraud__withdraw_W_Handle leads__schedule',
 'intercase_n_3__W_Assess potential fraud__schedule_W_Handle leads__schedule_W_Handle leads__start',
 'intercase_n_3__W_Assess potential fraud__start_W_Assess potential fraud__complete_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Assess potential fraud__start_W_Assess potential fraud__complete_W_Complete application__schedule',
 'intercase_n_3__W_Assess potential fraud__start_W_Assess potential fraud__suspend_W_Assess potential fraud__ate_abort',
 'intercase_n_3__W_Assess potential fraud__start_W_Assess potential fraud__suspend_W_Assess potential fraud__resume',
 'intercase_n_3__W_Assess potential fraud__start_W_Assess potential fraud__suspend_W_Call after offers__schedule',
 'intercase_n_3__W_Assess potential fraud__start_W_Assess potential fraud__suspend_W_Complete application__schedule',
 'intercase_n_3__W_Assess potential fraud__start_W_Assess potential fraud__suspend_W_Handle leads__schedule',
 'intercase_n_3__W_Assess potential fraud__start_W_Assess potential fraud__suspend_W_Validate application__schedule',
 'intercase_n_3__W_Assess potential fraud__start_W_Call after offers__schedule_W_Call after offers__start',
 'intercase_n_3__W_Assess potential fraud__start_W_Handle leads__schedule_W_Handle leads__start',
 'intercase_n_3__W_Assess potential fraud__start_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Assess potential fraud__suspend_W_Assess potential fraud__ate_abort_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Assess potential fraud__suspend_W_Assess potential fraud__ate_abort_W_Validate application__schedule',
 'intercase_n_3__W_Assess potential fraud__suspend_W_Assess potential fraud__resume_W_Assess potential fraud__complete',
 'intercase_n_3__W_Assess potential fraud__suspend_W_Assess potential fraud__resume_W_Assess potential fraud__start',
 'intercase_n_3__W_Assess potential fraud__suspend_W_Assess potential fraud__resume_W_Assess potential fraud__suspend',
 'intercase_n_3__W_Assess potential fraud__suspend_W_Assess potential fraud__resume_W_Call after offers__schedule',
 'intercase_n_3__W_Assess potential fraud__suspend_W_Assess potential fraud__resume_W_Complete application__schedule',
 'intercase_n_3__W_Assess potential fraud__suspend_W_Assess potential fraud__resume_W_Handle leads__schedule',
 'intercase_n_3__W_Assess potential fraud__suspend_W_Assess potential fraud__resume_W_Validate application__schedule',
 'intercase_n_3__W_Assess potential fraud__suspend_W_Call after offers__schedule_W_Call after offers__start',
 'intercase_n_3__W_Assess potential fraud__suspend_W_Complete application__schedule_W_Complete application__start',
 'intercase_n_3__W_Assess potential fraud__suspend_W_Handle leads__schedule_W_Handle leads__start',
 'intercase_n_3__W_Assess potential fraud__suspend_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Assess potential fraud__withdraw_W_Handle leads__schedule_W_Handle leads__start',
 'intercase_n_3__W_Call after offers__ate_abort_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Call after offers__ate_abort_W_Call after offers__schedule_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Call after offers__ate_abort_W_Call after offers__schedule_W_Call after offers__start',
 'intercase_n_3__W_Call after offers__ate_abort_W_Call after offers__schedule_W_Call after offers__withdraw',
 'intercase_n_3__W_Call after offers__ate_abort_W_Call after offers__schedule_W_Validate application__schedule',
 'intercase_n_3__W_Call after offers__ate_abort_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Call after offers__complete_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Call after offers__resume_W_Call after offers__start_W_Call after offers__suspend',
 'intercase_n_3__W_Call after offers__resume_W_Call after offers__suspend_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Call after offers__resume_W_Call after offers__suspend_W_Call after offers__ate_abort',
 'intercase_n_3__W_Call after offers__resume_W_Call after offers__suspend_W_Call after offers__resume',
 'intercase_n_3__W_Call after offers__resume_W_Call after offers__suspend_W_Shortened completion __resume',
 'intercase_n_3__W_Call after offers__resume_W_Call after offers__suspend_W_Validate application__schedule',
 'intercase_n_3__W_Call after offers__resume_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Call after offers__schedule_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Call after offers__schedule_W_Call after offers__start_W_Call after offers__complete',
 'intercase_n_3__W_Call after offers__schedule_W_Call after offers__start_W_Call after offers__suspend',
 'intercase_n_3__W_Call after offers__schedule_W_Call after offers__start_W_Shortened completion __schedule',
 'intercase_n_3__W_Call after offers__schedule_W_Call after offers__start_W_Shortened completion __suspend',
 'intercase_n_3__W_Call after offers__schedule_W_Call after offers__start_W_Validate application__schedule',
 'intercase_n_3__W_Call after offers__schedule_W_Call after offers__withdraw_W_Call after offers__schedule',
 'intercase_n_3__W_Call after offers__schedule_W_Call after offers__withdraw_W_Validate application__schedule',
 'intercase_n_3__W_Call after offers__schedule_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Call after offers__start_W_Call after offers__complete_W_Validate application__schedule',
 'intercase_n_3__W_Call after offers__start_W_Call after offers__suspend_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Call after offers__start_W_Call after offers__suspend_W_Call after offers__ate_abort',
 'intercase_n_3__W_Call after offers__start_W_Call after offers__suspend_W_Call after offers__resume',
 'intercase_n_3__W_Call after offers__start_W_Call after offers__suspend_W_Shortened completion __schedule',
 'intercase_n_3__W_Call after offers__start_W_Call after offers__suspend_W_Validate application__schedule',
 'intercase_n_3__W_Call after offers__start_W_Shortened completion __schedule_W_Shortened completion __start',
 'intercase_n_3__W_Call after offers__start_W_Shortened completion __suspend_W_Call after offers__suspend',
 'intercase_n_3__W_Call after offers__start_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Call after offers__suspend_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Call after offers__suspend_W_Call after offers__ate_abort_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Call after offers__suspend_W_Call after offers__ate_abort_W_Call after offers__schedule',
 'intercase_n_3__W_Call after offers__suspend_W_Call after offers__ate_abort_W_Validate application__schedule',
 'intercase_n_3__W_Call after offers__suspend_W_Call after offers__resume_W_Call after offers__start',
 'intercase_n_3__W_Call after offers__suspend_W_Call after offers__resume_W_Call after offers__suspend',
 'intercase_n_3__W_Call after offers__suspend_W_Call after offers__resume_W_Validate application__schedule',
 'intercase_n_3__W_Call after offers__suspend_W_Shortened completion __schedule_W_Shortened completion __start',
 'intercase_n_3__W_Call after offers__suspend_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Call after offers__withdraw_W_Call after offers__schedule_W_Call after offers__start',
 'intercase_n_3__W_Call after offers__withdraw_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Call incomplete files__ate_abort_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Call incomplete files__ate_abort_W_Call after offers__schedule_W_Call after offers__start',
 'intercase_n_3__W_Call incomplete files__ate_abort_W_Call incomplete files__schedule_W_Call incomplete files__start',
 'intercase_n_3__W_Call incomplete files__ate_abort_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Call incomplete files__complete_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Call incomplete files__complete_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Call incomplete files__resume_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Call incomplete files__resume_W_Call incomplete files__complete_W_Validate application__schedule',
 'intercase_n_3__W_Call incomplete files__resume_W_Call incomplete files__start_W_Call incomplete files__complete',
 'intercase_n_3__W_Call incomplete files__resume_W_Call incomplete files__start_W_Call incomplete files__suspend',
 'intercase_n_3__W_Call incomplete files__resume_W_Call incomplete files__start_W_Validate application__schedule',
 'intercase_n_3__W_Call incomplete files__resume_W_Call incomplete files__suspend_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Call incomplete files__resume_W_Call incomplete files__suspend_W_Call after offers__schedule',
 'intercase_n_3__W_Call incomplete files__resume_W_Call incomplete files__suspend_W_Call incomplete files__ate_abort',
 'intercase_n_3__W_Call incomplete files__resume_W_Call incomplete files__suspend_W_Call incomplete files__resume',
 'intercase_n_3__W_Call incomplete files__resume_W_Call incomplete files__suspend_W_Validate application__schedule',
 'intercase_n_3__W_Call incomplete files__resume_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Call incomplete files__schedule_W_Call incomplete files__start_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Call incomplete files__schedule_W_Call incomplete files__start_W_Call incomplete files__complete',
 'intercase_n_3__W_Call incomplete files__schedule_W_Call incomplete files__start_W_Call incomplete files__suspend',
 'intercase_n_3__W_Call incomplete files__schedule_W_Call incomplete files__start_W_Validate application__schedule',
 'intercase_n_3__W_Call incomplete files__start_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Call incomplete files__start_W_Call incomplete files__complete_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Call incomplete files__start_W_Call incomplete files__complete_W_Validate application__schedule',
 'intercase_n_3__W_Call incomplete files__start_W_Call incomplete files__suspend_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Call incomplete files__start_W_Call incomplete files__suspend_W_Call incomplete files__ate_abort',
 'intercase_n_3__W_Call incomplete files__start_W_Call incomplete files__suspend_W_Call incomplete files__resume',
 'intercase_n_3__W_Call incomplete files__start_W_Call incomplete files__suspend_W_Validate application__schedule',
 'intercase_n_3__W_Call incomplete files__start_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Call incomplete files__suspend_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Call incomplete files__suspend_W_Call after offers__schedule_W_Call after offers__start',
 'intercase_n_3__W_Call incomplete files__suspend_W_Call incomplete files__ate_abort_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Call incomplete files__suspend_W_Call incomplete files__ate_abort_W_Call after offers__schedule',
 'intercase_n_3__W_Call incomplete files__suspend_W_Call incomplete files__ate_abort_W_Call incomplete files__schedule',
 'intercase_n_3__W_Call incomplete files__suspend_W_Call incomplete files__ate_abort_W_Validate application__schedule',
 'intercase_n_3__W_Call incomplete files__suspend_W_Call incomplete files__resume_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Call incomplete files__suspend_W_Call incomplete files__resume_W_Call incomplete files__complete',
 'intercase_n_3__W_Call incomplete files__suspend_W_Call incomplete files__resume_W_Call incomplete files__start',
 'intercase_n_3__W_Call incomplete files__suspend_W_Call incomplete files__resume_W_Call incomplete files__suspend',
 'intercase_n_3__W_Call incomplete files__suspend_W_Call incomplete files__resume_W_Validate application__schedule',
 'intercase_n_3__W_Call incomplete files__suspend_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Complete application__ate_abort_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Complete application__ate_abort_W_Complete application__schedule_W_Complete application__start',
 'intercase_n_3__W_Complete application__complete_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Complete application__complete_W_Complete application__schedule_W_Complete application__start',
 'intercase_n_3__W_Complete application__resume_W_Call after offers__schedule_W_Call after offers__start',
 'intercase_n_3__W_Complete application__resume_W_Complete application__start_W_Call after offers__schedule',
 'intercase_n_3__W_Complete application__resume_W_Complete application__start_W_Complete application__suspend',
 'intercase_n_3__W_Complete application__resume_W_Complete application__suspend_W_Call after offers__schedule',
 'intercase_n_3__W_Complete application__resume_W_Complete application__suspend_W_Complete application__ate_abort',
 'intercase_n_3__W_Complete application__resume_W_Complete application__suspend_W_Complete application__resume',
 'intercase_n_3__W_Complete application__resume_W_Shortened completion __schedule_W_Shortened completion __start',
 'intercase_n_3__W_Complete application__schedule',
 'intercase_n_3__W_Complete application__schedule_W_Call after offers__schedule_W_Call after offers__start',
 'intercase_n_3__W_Complete application__schedule_W_Complete application__start',
 'intercase_n_3__W_Complete application__schedule_W_Complete application__start_W_Call after offers__schedule',
 'intercase_n_3__W_Complete application__schedule_W_Complete application__start_W_Complete application__complete',
 'intercase_n_3__W_Complete application__schedule_W_Complete application__start_W_Complete application__suspend',
 'intercase_n_3__W_Complete application__schedule_W_Complete application__start_W_Shortened completion __schedule',
 'intercase_n_3__W_Complete application__schedule_W_Shortened completion __schedule_W_Shortened completion __start',
 'intercase_n_3__W_Complete application__start_W_Call after offers__schedule_W_Call after offers__start',
 'intercase_n_3__W_Complete application__start_W_Complete application__complete_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Complete application__start_W_Complete application__complete_W_Complete application__schedule',
 'intercase_n_3__W_Complete application__start_W_Complete application__suspend_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Complete application__start_W_Complete application__suspend_W_Call after offers__schedule',
 'intercase_n_3__W_Complete application__start_W_Complete application__suspend_W_Complete application__ate_abort',
 'intercase_n_3__W_Complete application__start_W_Complete application__suspend_W_Complete application__resume',
 'intercase_n_3__W_Complete application__start_W_Complete application__suspend_W_Shortened completion __schedule',
 'intercase_n_3__W_Complete application__start_W_Shortened completion __schedule_W_Shortened completion __start',
 'intercase_n_3__W_Complete application__suspend_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Complete application__suspend_W_Call after offers__schedule_W_Call after offers__start',
 'intercase_n_3__W_Complete application__suspend_W_Complete application__ate_abort_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Complete application__suspend_W_Complete application__ate_abort_W_Complete application__schedule',
 'intercase_n_3__W_Complete application__suspend_W_Complete application__resume_W_Call after offers__schedule',
 'intercase_n_3__W_Complete application__suspend_W_Complete application__resume_W_Complete application__start',
 'intercase_n_3__W_Complete application__suspend_W_Complete application__resume_W_Complete application__suspend',
 'intercase_n_3__W_Complete application__suspend_W_Complete application__resume_W_Shortened completion __schedule',
 'intercase_n_3__W_Complete application__suspend_W_Shortened completion __schedule_W_Shortened completion __start',
 'intercase_n_3__W_Handle leads__complete_W_Handle leads__schedule_W_Handle leads__start',
 'intercase_n_3__W_Handle leads__resume_W_Complete application__schedule_W_Call after offers__schedule',
 'intercase_n_3__W_Handle leads__resume_W_Complete application__schedule_W_Complete application__start',
 'intercase_n_3__W_Handle leads__resume_W_Handle leads__complete_W_Handle leads__schedule',
 'intercase_n_3__W_Handle leads__resume_W_Handle leads__start_W_Complete application__schedule',
 'intercase_n_3__W_Handle leads__resume_W_Handle leads__start_W_Handle leads__suspend',
 'intercase_n_3__W_Handle leads__resume_W_Handle leads__suspend_W_Complete application__schedule',
 'intercase_n_3__W_Handle leads__resume_W_Handle leads__suspend_W_Handle leads__resume',
 'intercase_n_3__W_Handle leads__schedule',
 'intercase_n_3__W_Handle leads__schedule_W_Complete application__schedule',
 'intercase_n_3__W_Handle leads__schedule_W_Complete application__schedule_W_Call after offers__schedule',
 'intercase_n_3__W_Handle leads__schedule_W_Complete application__schedule_W_Complete application__start',
 'intercase_n_3__W_Handle leads__schedule_W_Complete application__schedule_W_Shortened completion __schedule',
 'intercase_n_3__W_Handle leads__schedule_W_Handle leads__start',
 'intercase_n_3__W_Handle leads__schedule_W_Handle leads__start_W_Complete application__schedule',
 'intercase_n_3__W_Handle leads__schedule_W_Handle leads__start_W_Handle leads__suspend',
 'intercase_n_3__W_Handle leads__schedule_W_Handle leads__withdraw',
 'intercase_n_3__W_Handle leads__schedule_W_Handle leads__withdraw_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Handle leads__start_W_Complete application__schedule_W_Call after offers__schedule',
 'intercase_n_3__W_Handle leads__start_W_Complete application__schedule_W_Complete application__start',
 'intercase_n_3__W_Handle leads__start_W_Handle leads__suspend_W_Complete application__schedule',
 'intercase_n_3__W_Handle leads__start_W_Handle leads__suspend_W_Handle leads__resume',
 'intercase_n_3__W_Handle leads__suspend_W_Complete application__schedule_W_Call after offers__schedule',
 'intercase_n_3__W_Handle leads__suspend_W_Complete application__schedule_W_Complete application__start',
 'intercase_n_3__W_Handle leads__suspend_W_Handle leads__resume_W_Complete application__schedule',
 'intercase_n_3__W_Handle leads__suspend_W_Handle leads__resume_W_Handle leads__complete',
 'intercase_n_3__W_Handle leads__suspend_W_Handle leads__resume_W_Handle leads__start',
 'intercase_n_3__W_Handle leads__suspend_W_Handle leads__resume_W_Handle leads__suspend',
 'intercase_n_3__W_Handle leads__withdraw_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Handle leads__withdraw_W_Assess potential fraud__schedule_W_Assess potential fraud__withdraw',
 'intercase_n_3__W_Handle leads__withdraw_W_Assess potential fraud__schedule_W_Handle leads__schedule',
 'intercase_n_3__W_Shortened completion __resume_W_Shortened completion __suspend_W_Shortened completion __resume',
 'intercase_n_3__W_Shortened completion __resume_W_Validate application__ate_abort_W_Call incomplete files__schedule',
 'intercase_n_3__W_Shortened completion __resume_W_Validate application__resume_W_Validate application__suspend',
 'intercase_n_3__W_Shortened completion __schedule_W_Shortened completion __start_W_Call after offers__resume',
 'intercase_n_3__W_Shortened completion __schedule_W_Shortened completion __start_W_Call after offers__schedule',
 'intercase_n_3__W_Shortened completion __schedule_W_Shortened completion __start_W_Call after offers__suspend',
 'intercase_n_3__W_Shortened completion __schedule_W_Shortened completion __start_W_Complete application__suspend',
 'intercase_n_3__W_Shortened completion __schedule_W_Shortened completion __start_W_Shortened completion __suspend',
 'intercase_n_3__W_Shortened completion __schedule_W_Shortened completion __start_W_Validate application__schedule',
 'intercase_n_3__W_Shortened completion __start_W_Call after offers__resume_W_Call after offers__suspend',
 'intercase_n_3__W_Shortened completion __start_W_Call after offers__schedule_W_Call after offers__start',
 'intercase_n_3__W_Shortened completion __start_W_Call after offers__suspend_W_Call after offers__resume',
 'intercase_n_3__W_Shortened completion __start_W_Call after offers__suspend_W_Validate application__schedule',
 'intercase_n_3__W_Shortened completion __start_W_Complete application__suspend_W_Call after offers__schedule',
 'intercase_n_3__W_Shortened completion __start_W_Complete application__suspend_W_Complete application__resume',
 'intercase_n_3__W_Shortened completion __start_W_Shortened completion __suspend_W_Call after offers__resume',
 'intercase_n_3__W_Shortened completion __start_W_Shortened completion __suspend_W_Complete application__start',
 'intercase_n_3__W_Shortened completion __start_W_Shortened completion __suspend_W_Complete application__suspend',
 'intercase_n_3__W_Shortened completion __start_W_Shortened completion __suspend_W_Shortened completion __resume',
 'intercase_n_3__W_Shortened completion __start_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Shortened completion __suspend_W_Call after offers__resume_W_Call after offers__suspend',
 'intercase_n_3__W_Shortened completion __suspend_W_Call after offers__suspend_W_Call after offers__resume',
 'intercase_n_3__W_Shortened completion __suspend_W_Complete application__start_W_Complete application__suspend',
 'intercase_n_3__W_Shortened completion __suspend_W_Complete application__suspend_W_Complete application__resume',
 'intercase_n_3__W_Shortened completion __suspend_W_Shortened completion __resume_W_Shortened completion __suspend',
 'intercase_n_3__W_Validate application__ate_abort_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Validate application__ate_abort_W_Call incomplete files__schedule_W_Call incomplete files__start',
 'intercase_n_3__W_Validate application__ate_abort_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Validate application__complete_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Validate application__complete_W_Call incomplete files__schedule_W_Call incomplete files__start',
 'intercase_n_3__W_Validate application__complete_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Validate application__resume_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Validate application__resume_W_Call incomplete files__schedule_W_Call incomplete files__start',
 'intercase_n_3__W_Validate application__resume_W_Validate application__complete_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Validate application__resume_W_Validate application__complete_W_Call incomplete files__schedule',
 'intercase_n_3__W_Validate application__resume_W_Validate application__start_W_Call incomplete files__schedule',
 'intercase_n_3__W_Validate application__resume_W_Validate application__start_W_Validate application__complete',
 'intercase_n_3__W_Validate application__resume_W_Validate application__start_W_Validate application__suspend',
 'intercase_n_3__W_Validate application__resume_W_Validate application__suspend_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Validate application__resume_W_Validate application__suspend_W_Call incomplete files__schedule',
 'intercase_n_3__W_Validate application__resume_W_Validate application__suspend_W_Shortened completion __resume',
 'intercase_n_3__W_Validate application__resume_W_Validate application__suspend_W_Validate application__ate_abort',
 'intercase_n_3__W_Validate application__resume_W_Validate application__suspend_W_Validate application__resume',
 'intercase_n_3__W_Validate application__schedule_W_Validate application__start_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Validate application__schedule_W_Validate application__start_W_Call incomplete files__schedule',
 'intercase_n_3__W_Validate application__schedule_W_Validate application__start_W_Validate application__complete',
 'intercase_n_3__W_Validate application__schedule_W_Validate application__start_W_Validate application__suspend',
 'intercase_n_3__W_Validate application__start_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Validate application__start_W_Call incomplete files__schedule_W_Call incomplete files__start',
 'intercase_n_3__W_Validate application__start_W_Validate application__complete_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Validate application__start_W_Validate application__complete_W_Call incomplete files__schedule',
 'intercase_n_3__W_Validate application__start_W_Validate application__complete_W_Validate application__schedule',
 'intercase_n_3__W_Validate application__start_W_Validate application__suspend_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Validate application__start_W_Validate application__suspend_W_Call incomplete files__schedule',
 'intercase_n_3__W_Validate application__start_W_Validate application__suspend_W_Shortened completion __resume',
 'intercase_n_3__W_Validate application__start_W_Validate application__suspend_W_Validate application__ate_abort',
 'intercase_n_3__W_Validate application__start_W_Validate application__suspend_W_Validate application__resume',
 'intercase_n_3__W_Validate application__suspend_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Validate application__suspend_W_Call incomplete files__schedule_W_Call incomplete files__start',
 'intercase_n_3__W_Validate application__suspend_W_Shortened completion __resume_W_Validate application__ate_abort',
 'intercase_n_3__W_Validate application__suspend_W_Shortened completion __resume_W_Validate application__resume',
 'intercase_n_3__W_Validate application__suspend_W_Validate application__ate_abort_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Validate application__suspend_W_Validate application__ate_abort_W_Call incomplete files__schedule',
 'intercase_n_3__W_Validate application__suspend_W_Validate application__ate_abort_W_Validate application__schedule',
 'intercase_n_3__W_Validate application__suspend_W_Validate application__resume_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Validate application__suspend_W_Validate application__resume_W_Call incomplete files__schedule',
 'intercase_n_3__W_Validate application__suspend_W_Validate application__resume_W_Validate application__complete',
 'intercase_n_3__W_Validate application__suspend_W_Validate application__resume_W_Validate application__start',
 'intercase_n_3__W_Validate application__suspend_W_Validate application__resume_W_Validate application__suspend'
]

In [4]:
n_processes = 32
batch_size = 24
N = 1000

In [5]:
test_data

,Action_start,org:resource_start,concept:name,EventOrigin_start,EventID_start,lifecycle:transition_start,time:timestamp_start,case:LoanGoal_start,case:ApplicationType_start,case:concept:name,...,intercase_n_3__W_Validate application__suspend_W_Shortened completion __resume_W_Validate application__ate_abort,intercase_n_3__W_Validate application__suspend_W_Shortened completion __resume_W_Validate application__resume,intercase_n_3__W_Validate application__suspend_W_Validate application__ate_abort_W_Assess potential fraud__schedule,intercase_n_3__W_Validate application__suspend_W_Validate application__ate_abort_W_Call incomplete files__schedule,intercase_n_3__W_Validate application__suspend_W_Validate application__ate_abort_W_Validate application__schedule,intercase_n_3__W_Validate application__suspend_W_Validate application__resume_W_Assess potential fraud__schedule,intercase_n_3__W_Validate application__suspend_W_Validate application__resume_W_Call incomplete files__schedule,intercase_n_3__W_Validate application__suspend_W_Validate application__resume_W_Validate application__complete,intercase_n_3__W_Validate application__suspend_W_Validate application__resume_W_Validate application__start,intercase_n_3__W_Validate application__suspend_W_Validate application__resume_W_Validate application__suspend
6,Obtained,User_5,W_Call after offers__resume,Workflow,Workitem_1000019350,resume,2016-02-13 15:34:25.947000+00:00,Home improvement,New credit,Application_2110398373,...,0,0,0,0,0,0,0,0,0,9
18,Released,User_38,W_Call after offers__suspend,Workflow,Workitem_1000053568,suspend,2016-03-16 16:18:16.350000+00:00,Existing loan takeover,New credit,Application_1996091858,...,0,0,0,0,0,0,0,0,0,30
20,Obtained,User_29,W_Call incomplete files__start,Workflow,Workitem_1000065077,start,2016-11-29 15:46:43.081000+00:00,Existing loan takeover,New credit,Application_1196087089,...,0,0,0,0,0,0,0,0,0,11
24,Released,User_100,W_Validate application__suspend,Workflow,Workitem_1000078627,suspend,2016-04-04 10:28:20.976000+00:00,Car,New credit,Application_519522134,...,0,0,0,0,0,0,0,0,0,25
28,Obtained,User_52,W_Complete application__start,Workflow,Workitem_1000092342,start,2016-09-06 13:25:25.463000+00:00,Car,New credit,Application_159861644,...,0,0,0,0,0,0,0,0,0,13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
619775,Released,User_10,W_Complete application__suspend,Workflow,Workitem_999975609,suspend,2016-09-05 16:26:51.673000+00:00,Car,New credit,Application_1386592498,...,0,0,0,0,0,0,0,0,0,15
619776,Released,User_29,W_Validate application__suspend,Workflow,Workitem_999981856,suspend,2016-01-18 14:37:21.843000+00:00,Existing loan takeover,New credit,Application_947906225,...,0,0,0,0,0,0,0,0,0,15
619778,Obtained,User_28,W_Call incomplete files__resume,Workflow,Workitem_999990412,resume,2016-04-01 18:08:48.145000+00:00,Car,New credit,Application_883995052,...,0,0,0,0,0,0,0,0,0,12
619780,Created,User_118,W_Validate application__schedule,Workflow,Workitem_99999173,schedule,2016-04-13 07:49:14.822000+00:00,Not speficied,New credit,Application_52539020,...,0,0,0,0,0,0,0,0,0,40


In [10]:
from pyinstrument import Profiler

prof = Profiler()
prof.start()

try:
    evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['ARSDACRCII1'],
                                                    SampleOutcomes_QuantileRegression_ARSDACRCII, {
                                                            'case_id_key' : 'case:concept:name',
                                                            'activity_key' : 'concept:name',
                                                            'resource_key' : 'org:resource_start',
                                                            'inter_instance_column_names' : ii1
                                                        },
                                        test_data, n_processes=n_processes, batch_size=batch_size, n=N)
    likelihoods_A = evaluator_A.sample_cases(False, False)
except KeyboardInterrupt:
    pass
finally:
    prof.stop()
    with open("profile.html", "w") as f:
        f.write(prof.output_html())


  0%|          | 1/6068 [02:56<298:11:06, 176.94s/it]


In [12]:
import cProfile

profiler = cProfile.Profile()
profiler.enable()

try:
    evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['ARSDACRCII1'],
                                                    SampleOutcomes_QuantileRegression_ARSDACRCII, {
                                                            'case_id_key' : 'case:concept:name',
                                                            'activity_key' : 'concept:name',
                                                            'resource_key' : 'org:resource_start',
                                                            'inter_instance_column_names' : ii1
                                                        },
                                        test_data, n_processes=n_processes, batch_size=batch_size, n=N)
    likelihoods_A = evaluator_A.sample_cases(False, False)
except KeyboardInterrupt:
    pass
finally:
    profiler.disable()
    profiler.print_stats(sort='calls')  


  0%|          | 1/6068 [02:15<227:42:47, 135.12s/it]

         355214022 function calls (353272958 primitive calls) in 135.023 seconds

   Ordered by: call count

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
125429275/125429271    8.674    0.000   11.183    0.000 {built-in method builtins.isinstance}
 64575274    8.625    0.000   11.840    0.000 base.py:3816(<genexpr>)
  7042106    0.376    0.000    0.376    0.000 {built-in method builtins.callable}
  7041470    3.250    0.000    4.900    0.000 indexing.py:2801(check_dict_or_set_indexers)
  7041470    1.015    0.000    1.391    0.000 common.py:372(apply_if_callable)
  7040452    8.687    0.000   28.065    0.000 base.py:3784(get_loc)
  7040452    0.403    0.000    0.403    0.000 base.py:6718(_maybe_cast_indexer)
  7040448    2.548    0.000   31.336    0.000 series.py:1232(_get_value)
  7040448    6.178    0.000   44.393    0.000 series.py:1107(__getitem__)
6639516/4698464    0.711    0.000    1.120    0.000 {built-in method builtins.len}
  6464886    3.304    0.0

In [ ]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

In [ ]:
np.mean(get_pscores(likelihoods_A))

In [ ]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['ARSDACRCII3'],
                                                   SampleOutcomes_QuantileRegression_ARSDACRCII, {
                                                        'case_id_key' : 'case:concept:name',
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                        'inter_instance_column_names' : ii3
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

In [ ]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

In [ ]:
np.mean(get_pscores(likelihoods_A))

In [ ]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['ARSDII1'],
                                                   SampleOutcomes_QuantileRegression_ARSDII, {
                                                        'case_id_key' : 'case:concept:name',
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                        'inter_instance_column_names' : ii1
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

In [ ]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

In [ ]:
np.mean(get_pscores(likelihoods_A))

In [ ]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['ARSDII3'],
                                                   SampleOutcomes_QuantileRegression_ARSDII, {
                                                        'case_id_key' : 'case:concept:name',
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                        'inter_instance_column_names' : ii3
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

In [ ]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

In [ ]:
np.mean(get_pscores(likelihoods_A))